In [1]:
import modules.data as d
import modules.model as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn

from pathlib import Path


# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# define experiment
def run_experiment(data, model_class, model_kwargs):
    # data module
    dmod = t.DataModule(
        data.X.squeeze(),
        data.y,
        generator,
        batch_size=256
    )

    # define, run experiment
    tmod = t.MultiTrainingModule(
        model_class=model_class,
        model_kwargs=model_kwargs,
        
        training_class=t.ClassifierTrainingModule,
        training_kwargs={
            'data_module':dmod,
            'loss_fn':nn.CrossEntropyLoss(dmod.class_weights),
            'optimizer':torch.optim.Adam,
            'num_epochs':100,
            'report_metrics':['tot_loss', 'accuracy'],
        },

        num_trials=10,
        comment='mlpc_mt_test'
    )

    return (dmod, tmod)

*** Device() ***
device = cuda:2



---

In [2]:
# get data
brca = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    # max_subset=120,
)

# expt
brca_MLPc = run_experiment(
    data=brca,
    model_class=m.MLPClassifier,
    model_kwargs={
            'in_features':brca.num_nodes,
            'out_features':brca.num_labels,
            'mlp_kwargs':{
                'hidden_dims':[64],
                'bias':True,
                'act_fn':nn.ReLU()
            },
        },
)

brca_GCNc = run_experiment(
    data=brca,
    model_class=m.GCNClassifier,
    model_kwargs={
        'in_features':brca.num_features,
        'out_features':brca.num_labels,
        'relation':brca.relation,
        'num_nodes':brca.num_nodes,
        'gcn_kwargs':{'end_fn':nn.LeakyReLU()}
    },
)

**** Data() ****
log0_method      str
gene_counts      (4383, 1184)     DataFrame
metadata         (1184, 3)        DataFrame
relation         (75939, 19)      DataFrame
node_id_map      dict
masks            305              list
X                (1184, 4383, 1)  Tensor (cuda:2)
y                (1184, 6)        Tensor (cuda:2)
y_labels         6                list
num_samples      1184             int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



100%|██████████| 100/100 [00:02<00:00, 45.62it/s, Epoch 99      Train: tot_loss=0.1713    accuracy=0.9976        Val: tot_loss=8.4749    accuracy=0.8192]


Test	 tot_loss=0.9662    accuracy=0.9209



100%|██████████| 100/100 [00:01<00:00, 50.36it/s, Epoch 99      Train: tot_loss=0.1014    accuracy=1.0000        Val: tot_loss=9.9572    accuracy=0.8305]


Test	 tot_loss=1.1140    accuracy=0.8927



...

Test	 tot_loss=0.9406    accuracy=0.9209



100%|██████████| 100/100 [00:01<00:00, 50.70it/s, Epoch 99      Train: tot_loss=0.1256    accuracy=0.9976        Val: tot_loss=9.2159    accuracy=0.8249]


Test	 tot_loss=1.0511    accuracy=0.9153



  0%|          | 0/100 [00:00<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (4383x4383 and 256x4383)

In [3]:
brca.relation

,pathway_name,idx1,idx2,activation,binding/association,compound,dephosphorylation,dissociation,expression,glycosylation,indirect,indirect effect,inhibition,methylation,missing interaction,phosphorylation,repression,state change,ubiquitination
0,path:hsa04975,0,2,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,path:hsa04975,0,3,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
2,path:hsa04975,0,4,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
3,path:hsa04975,1,2,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
4,path:hsa04975,1,3,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75934,path:hsa00190,2683,4381,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75935,path:hsa00190,2683,4382,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75936,path:hsa00190,3356,4380,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75937,path:hsa00190,3356,4381,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


In [5]:
adj = m._get_adjacency(brca.relation, brca.num_nodes)
adj.shape

torch.Size([4383, 4383])

In [7]:
brca.X.shape

torch.Size([1184, 4383, 1])

In [9]:
torch.matmul(adj, brca.X).shape

torch.Size([1184, 4383, 1])